# Model Training, Tuning, and Ensembling

In this notebook, we will compare six model families and construct a stacking ensemble. The modeling process is separated into three chronological stages:

1. Tune every model with expanding-season validation through 2023
2. Use 2024 to select between the individual models and the ensemble and to calibrate prediction intervals
3. Fit the final system through 2024 without viewing the 2025 targets

Margin and total are modeled separately. The 2025 season remains untouched until the final evaluation notebook.

In [1]:
from pathlib import Path
import sys

# Allow the notebook to run from either the project root or the notebooks folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## Imports and Data

In [2]:
import numpy as np
import pandas as pd

from src.config import EVALUATION_DIR, FINAL_TEST_SEASON, MODEL_DIR, PROCESSED_DIR
from src.features import get_model_features
from src.models import (
    add_score_predictions,
    build_model_bundle,
    compare_base_models_and_stack,
    expanding_season_folds,
    fit_model_collection,
    fit_stacking_model,
    generate_oof_predictions,
    save_model_bundle,
    tune_models,
)

pd.set_option('display.max_colwidth', 120)

modeling_table = pd.read_csv(
    PROCESSED_DIR / 'modeling_table.csv',
    parse_dates=['start_date'],
    low_memory=False,
)
completed = modeling_table[
    modeling_table['completed'].eq(True) &
    modeling_table['actual_margin'].notna() &
    modeling_table['actual_total'].notna()
].sort_values(['start_date', 'game_id']).reset_index(drop=True)

features = get_model_features(completed)
development = completed[completed['season'] < FINAL_TEST_SEASON].reset_index(drop=True)
tuning_data = development[development['season'] <= 2023].reset_index(drop=True)
selection_data = development[development['season'].eq(2024)].reset_index(drop=True)

print('Hyperparameter tuning games:', len(tuning_data))
print('2024 selection and calibration games:', len(selection_data))
print('Features:', len(features))
print('Untouched final test season:', FINAL_TEST_SEASON)

Hyperparameter tuning games: 7479
2024 selection and calibration games: 798
Features: 161
Untouched final test season: 2025


## Why These Models Were Selected

The comparison includes the four original model families and two additional nonlinear models.

- **Ridge Regression** is a stable linear baseline when many related features are included.
- **Elastic Net** can shrink or remove less useful features while retaining groups of correlated predictors.
- **Random Forest** uses bagging to estimate nonlinear relationships and interactions without requiring feature scaling.
- **Histogram Gradient Boosting** builds trees sequentially and is often effective for structured tabular data.
- **RBF Support Vector Regression** was added because it can learn smooth nonlinear boundaries in a high-dimensional feature space. Scaling is included in its pipeline.
- **Neural Network** was added to test whether several hidden layers can learn interactions among efficiency, team strength, and game-context variables. Dropout, weight decay, Huber loss, and early stopping are used to control overfitting.

Logistic regression and Naive Bayes are classification methods, while the primary targets in this project are continuous scores. They are therefore not forced into the main comparison. Winner accuracy is still reported after the predicted margin is converted to a winner.

## Expanding-Season Validation

In [3]:
tuning_folds = expanding_season_folds(
    tuning_data,
    first_validation_season=2020,
    last_validation_season=2023,
)

fold_summary = pd.DataFrame([
    {
        'validation season': season,
        'training games': len(train_index),
        'validation games': len(validation_index),
    }
    for train_index, validation_index, season in tuning_folds
])
fold_summary

,validation season,training games,validation games
0,2020,4607,534
1,2021,5141,770
2,2022,5911,776
3,2023,6687,792


For each fold, all training games occurred before the validation season. Imputation and scaling are contained inside each pipeline, so they are also fitted using only the training portion of a fold.

The full search is the default. It evaluates 52 hyperparameter settings for each target across four future-season folds. The neural network automatically uses a CUDA GPU when one is available. `FAST_MODE = True` is provided only as a short code check; it should not be used for the final reported results.

In [4]:
FAST_MODE = False

## Hyperparameter Tuning

In [5]:
margin_results, margin_estimators = tune_models(
    data=tuning_data,
    features=features,
    target='actual_margin',
    folds=tuning_folds,
    fast_mode=FAST_MODE,
)

margin_summary = (
    margin_results.groupby('model')['mean_validation_mae']
    .min()
    .sort_values()
    .to_frame()
)
margin_summary

Tuning Ridge: 4 parameter settings


  best validation MAE: 13.270 (0.0 minutes)
Tuning Elastic Net: 12 parameter settings


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.937e+05, tolerance: 2.109e+02
  model = cd_fast.enet_coordinate_descent(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.429e+05, tolerance: 2.342e+02
  model = cd_fast.enet_coordinate_descent(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.589e+05, tolerance: 2.677e+02
  model = cd_fast.enet_coordinate_descent(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.885e+05, tolerance: 2.986e+02
  model = cd_fast.enet_coordinate_descent(


... (12 outputs left out)

  best validation MAE: 13.096 (0.8 minutes)
Tuning RBF Support Vector Regression: 12 parameter settings


  best validation MAE: 13.376 (1.1 minutes)
Tuning Neural Network: 8 parameter settings


  best validation MAE: 13.264 (0.4 minutes)


,mean_validation_mae
model,
Gradient Boosting,13.095576
Random Forest,13.128144
Elastic Net,13.131847
Neural Network,13.263966
Ridge,13.270472
RBF Support Vector Regression,13.376062


In [6]:
total_results, total_estimators = tune_models(
    data=tuning_data,
    features=features,
    target='actual_total',
    folds=tuning_folds,
    fast_mode=FAST_MODE,
)

total_summary = (
    total_results.groupby('model')['mean_validation_mae']
    .min()
    .sort_values()
    .to_frame()
)
total_summary

Tuning Ridge: 4 parameter settings


  best validation MAE: 13.091 (0.0 minutes)
Tuning Elastic Net: 12 parameter settings


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.992e+05, tolerance: 1.610e+02
  model = cd_fast.enet_coordinate_descent(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.015e+05, tolerance: 1.791e+02
  model = cd_fast.enet_coordinate_descent(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.624e+05, tolerance: 2.019e+02
  model = cd_fast.enet_coordinate_descent(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.850e+05, tolerance: 2.259e+02
  model = cd_fast.enet_coordinate_descent(


... (12 outputs left out)

  best validation MAE: 13.194 (0.8 minutes)
Tuning RBF Support Vector Regression: 12 parameter settings


  best validation MAE: 13.180 (1.1 minutes)
Tuning Neural Network: 8 parameter settings


  best validation MAE: 13.122 (0.4 minutes)


,mean_validation_mae
model,
Elastic Net,13.076474
Ridge,13.090726
Neural Network,13.121526
Random Forest,13.162886
RBF Support Vector Regression,13.179506
Gradient Boosting,13.193622


## Out-of-Fold Stacking

A stacking ensemble uses the predictions from the six base models as inputs to a second model. The second model is a nonnegative linear regression. This keeps the combination interpretable and prevents a model from receiving a negative weight solely to compensate for noise.

The stacker is trained only on out-of-fold predictions. In other words, each prediction used to learn the ensemble weights was produced by a base model that had not seen that season.

In [7]:
margin_oof = generate_oof_predictions(
    tuning_data, features, 'actual_margin', tuning_folds, margin_estimators
)
total_oof = generate_oof_predictions(
    tuning_data, features, 'actual_total', tuning_folds, total_estimators
)

margin_model_names = list(margin_estimators)
total_model_names = list(total_estimators)
margin_stacker_2023 = fit_stacking_model(margin_oof, margin_model_names)
total_stacker_2023 = fit_stacking_model(total_oof, total_model_names)

stacking_weights = pd.DataFrame({
    'model': margin_model_names,
    'margin_weight': margin_stacker_2023.coef_,
    'total_weight': total_stacker_2023.coef_,
})
stacking_weights

OOF fold 1: predicting 2020


OOF fold 2: predicting 2021


OOF fold 3: predicting 2022


OOF fold 4: predicting 2023


OOF fold 1: predicting 2020


OOF fold 2: predicting 2021


OOF fold 3: predicting 2022


OOF fold 4: predicting 2023


,model,margin_weight,total_weight
0,Ridge,0.000000,0.280357
1,Elastic Net,0.258036,0.149143
2,Random Forest,0.189279,0.237438
3,Gradient Boosting,0.357112,0.196479
4,RBF Support Vector Regression,0.134407,0.171716
5,Neural Network,0.064586,0.033218


## 2024 Model and Ensemble Selection

In [8]:
margin_models_2023 = fit_model_collection(
    tuning_data, features, margin_estimators, 'actual_margin'
)
total_models_2023 = fit_model_collection(
    tuning_data, features, total_estimators, 'actual_total'
)

margin_selection, margin_selection_predictions = compare_base_models_and_stack(
    selection_data,
    features,
    margin_models_2023,
    margin_stacker_2023,
    'actual_margin',
)
total_selection, total_selection_predictions = compare_base_models_and_stack(
    selection_data,
    features,
    total_models_2023,
    total_stacker_2023,
    'actual_total',
)

margin_primary = margin_selection.iloc[0]['model']
total_primary = total_selection.iloc[0]['model']
print('Selected margin system:', margin_primary)
print('Selected total system:', total_primary)
display(margin_selection)
display(total_selection)

Fitting Ridge
Fitting Elastic Net


Fitting Random Forest


Fitting Gradient Boosting


Fitting RBF Support Vector Regression


Fitting Neural Network


Fitting Ridge
Fitting Elastic Net


... (3 outputs left out)

Fitting Neural Network


Selected margin system: Random Forest
Selected total system: Stacking Ensemble


,model,selection_mae
0,Random Forest,12.919869
1,Stacking Ensemble,12.968525
2,Elastic Net,12.989815
3,Gradient Boosting,13.056261
4,Neural Network,13.199643
5,Ridge,13.208497
6,RBF Support Vector Regression,13.264540


,model,selection_mae
0,Stacking Ensemble,12.781760
1,Gradient Boosting,12.785294
2,Elastic Net,12.797656
3,Ridge,12.827708
4,Neural Network,12.861240
5,RBF Support Vector Regression,12.898464
6,Random Forest,12.924910


The individual models and the ensemble are treated equally in this step. The ensemble is retained as the primary system only if it earns that choice on the later 2024 season. These predictions also provide an honest calibration set for the prediction intervals and the later market hybrid.

In [9]:
selection_bundle = {
    'features': features,
    'margin_models': margin_models_2023,
    'total_models': total_models_2023,
    'margin_stacker': margin_stacker_2023,
    'total_stacker': total_stacker_2023,
    'margin_model_names': margin_model_names,
    'total_model_names': total_model_names,
    'margin_primary': margin_primary,
    'total_primary': total_primary,
}
calibration_predictions = add_score_predictions(selection_data, selection_bundle)
calibration_predictions.to_csv(EVALUATION_DIR / 'calibration_predictions.csv', index=False)

margin_results.to_csv(EVALUATION_DIR / 'margin_validation_results.csv', index=False)
total_results.to_csv(EVALUATION_DIR / 'total_validation_results.csv', index=False)
margin_selection.to_csv(EVALUATION_DIR / 'margin_selection_2024.csv', index=False)
total_selection.to_csv(EVALUATION_DIR / 'total_selection_2024.csv', index=False)
stacking_weights.to_csv(EVALUATION_DIR / 'stacking_weights_through_2023.csv', index=False)

## Final Training Through 2024

In [10]:
# Add the untouched 2024 base-model predictions to the earlier OOF predictions.
margin_2024_oof = pd.DataFrame({
    'row_index': selection_data.index,
    'season': 2024,
    'actual': selection_data['actual_margin'].to_numpy(),
    **{name: margin_selection_predictions[name].to_numpy() for name in margin_model_names},
})
total_2024_oof = pd.DataFrame({
    'row_index': selection_data.index,
    'season': 2024,
    'actual': selection_data['actual_total'].to_numpy(),
    **{name: total_selection_predictions[name].to_numpy() for name in total_model_names},
})

final_margin_oof = pd.concat([margin_oof, margin_2024_oof], ignore_index=True)
final_total_oof = pd.concat([total_oof, total_2024_oof], ignore_index=True)
final_margin_stacker = fit_stacking_model(final_margin_oof, margin_model_names)
final_total_stacker = fit_stacking_model(final_total_oof, total_model_names)

final_bundle = build_model_bundle(
    training_data=development,
    features=features,
    margin_estimators=margin_estimators,
    total_estimators=total_estimators,
    margin_stacker=final_margin_stacker,
    total_stacker=final_total_stacker,
    margin_primary=margin_primary,
    total_primary=total_primary,
)
final_bundle['tuning_last_season'] = 2023
final_bundle['selection_season'] = 2024
final_bundle['final_test_season'] = FINAL_TEST_SEASON
save_model_bundle(final_bundle, MODEL_DIR / 'statistics_only_models.joblib')

final_weights = pd.DataFrame({
    'model': margin_model_names,
    'margin_weight': final_margin_stacker.coef_,
    'total_weight': final_total_stacker.coef_,
})
final_weights.to_csv(EVALUATION_DIR / 'final_stacking_weights.csv', index=False)
final_weights

Fitting Ridge
Fitting Elastic Net


Fitting Random Forest


Fitting Gradient Boosting


Fitting RBF Support Vector Regression


Fitting Neural Network


Fitting Ridge
Fitting Elastic Net


Fitting Random Forest


Fitting Gradient Boosting


Fitting RBF Support Vector Regression


Fitting Neural Network


,model,margin_weight,total_weight
0,Ridge,0.000000,0.400057
1,Elastic Net,0.146296,0.080685
2,Random Forest,0.360112,0.127109
3,Gradient Boosting,0.234813,0.274293
4,RBF Support Vector Regression,0.109751,0.120207
5,Neural Network,0.157399,0.000000


## Conclusion

Every base model completed the same lifecycle: preprocessing inside a pipeline, a bounded hyperparameter search, expanding-season validation, later-season selection, and refitting on all development data. The ensemble weights were learned from out-of-fold predictions rather than fitted values. The saved bundle contains all six models, both stackers, and the selected primary system for each target.